# Benchmark: Split Flow

In [1]:
import rioxarray as rxr
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0,"/home/jovyan/shared/Libraries")
import victor
import cartopy.crs as ccrs
import subprocess
import os
import pandas as pd
import xarray as xr
import rasterio as rio

#### This benchmark is based on an experiment performed at the Syracuse University Lava Project that involved pouring molten basalt at a constant supply rate onto a sloping plane (Dietterich et al., [2015](https://appliedvolc.biomedcentral.com/articles/10.1186/s13617-017-0061-x#ref-CR20)).

This simulation captures both thermal effects and their impacts on flow rheology. This experiment also allows us to compare the flow propagation and surface temperature between the numerical simulations and the experimental data.

We encourage changing the volume betweens runs to compare the physical or stochastic reactions at various scales. It is also recommended to change the slope of the bed to measure various internal friction effects. Finally, the angle of the obstacle, which can dictate if/how the separated flows recombine, 

In [2]:
bed_slope = 14

volume = 7.7e-4

split_angle = 90

#### The following cells should not be edited, unless you have extensive knowledge of the model.

The following cell sets up common parameters. Following this, the next 4 cells format and generate input files for each respective model, at parameters considered reasonable for this benchmark.

In [14]:
r[0,150,70]

<xarray.DataArray ()> Size: 4B
[1 values with dtype=float32]
Coordinates:
    band         int64 8B 1
    x            float64 8B 0.3535
    y            float64 8B 0.005
    spatial_ref  int64 8B 0
Attributes:
    scale_factor:  1.0
    add_offset:    0.0

In [17]:
easting = 0
northing = 0
scale = volume/7.7e-4

f = open("DEMs_Lava_Benchmark/slope.asc","r+")
header = f.readlines()
header[3] = f"""xllcorner {-2*scale}\n"""
header[4] = f"""yllcorner {-1.5*scale}\n"""
header[5] = f"""cellsize {0.01*scale}\n"""
f.seek(0)
f.writelines(header)
f.truncate()
f.close()

r = rxr.open_rasterio("DEMs_Lava_Benchmark/slope.asc")
height = r.rio.resolution()[0]*501*np.sin(np.deg2rad(bed_slope))
slope = np.linspace(height,0,501)
stacked = np.tile(slope,(301,1))
r[0,:,:] = stacked
y = 150
x = 70
for i in range(14):
    j = int(np.round(np.sin(np.deg2rad(split_angle))))
    r[0,y+j,x+j] = 5000
    r[0,y-j,x+j] = 5000
# r[0,:,:] = stacked
r.rio.to_raster('Benchmark_Split.asc')
xll, yll = int(r.x.min()), int(r.y.min())
coordinates = np.array([int(easting),int(northing)])

### Molasses: Additional Notes

Molasses input parameters are stored in `custom_molasses.conf` in the `Molasses` folder. The most impactful change would be that of changing the minimum and maximum residual, as the cellular automata structure will cause a much larger or smaller area to be impacted. As a stochastic model, no physical parameters are used.

In [ ]:
os.chdir('/home/jovyan/Lava Benchmark/Molasses')

f = open("custom_molasses.conf","r+")
conf = f.readlines()
conf[2] = f"""MIN_RESIDUAL = {7e-6*scale}\n"""
conf[3] = f"""MAX_RESIDUAL ={1e-3*scale}\n"""
conf[4] = f"""MIN_TOTAL_VOLUME = {str(volume)}\n"""
conf[5] = f"""MAX_TOTAL_VOLUME = {str(volume)}\n"""
conf[6] = f"""MIN_PULSE_VOLUME = {float(volume)/100}\n"""
conf[7] = f"""MAX_PULSE_VOLUME = {float(volume)/100}\n"""
conf[10] = f"""DEM_FILE = {../dem}\n"""
f.seek(0)
f.writelines(conf)
f.truncate()
f.close()

f=open("events.in","w")
f.write(f"""{easting},{northing}""")
f.close()

subprocess.run("molasses custom_molasses.conf",shell=True)

victor.convert_molasses("molasses_split")

### MrLavaLoba/Flowy: Additional Notes

MrLavaLoba/Flowy (a drop in C++ replacement) input parameters are stored in `input.toml` in the `flowy` folder. The most impactful changes would be those such as the thickening parameter, lobe exponent, or max slope probability. Other parameters not mentioned will also affect the creation and placement of lobes. As a stochastic model, no physical parameters are specified by the user.

In [ ]:
os.chdir('/home/jovyan/Lava Benchmark/flowy')
g=open("input.toml","r+")
inp = g.readlines()
inp[1] = f"""run_name ='Split'\n"""
inp[6] = f"""source = '../{dem}'\n"""
inp[21] = """vent_flag = 0\n"""
inp[24] = f"""x_vent = [{int(easting)}]\n"""
inp[25] = f"""y_vent = [{int(northing)}]\n"""
inp[63] = f"""total_volume = {volume}\n"""
g.seek(0)
g.writelines(inp)
g.truncate()
g.close()

subprocess.run("./flowy input.toml",shell=True)

In [ ]:
os.chdir('/home/jovyan/Lava Benchmark/IMEX_LavaFlow')
radius = 2e-3/scale
h=open("IMEX_LavaFlow.inp","r+")
inp = h.readlines()
inp[1] = 'RUN_NAME="Split           ",\n'
inp[4] = f""" T_END=  600,\n"""
inp[5] = f""" DT_OUTPUT=  120,\n"""
inp[13] = f""" X0= {xll}D0     ,\n"""
inp[14] = f""" Y0= {yll}D0      ,\n"""
inp[15] = " COMP_CELLS_X=  500,\n"
inp[16] = " COMP_CELLS_Y=  300,\n"
inp[17] = f""" CELL_SIZE = {res}D0 ,\n"""
inp[103] = f"""  X_SOURCE=  {easting}.0D0 ,\n"""
inp[104] = f"""  Y_SOURCE=  {northing}.0D0 ,\n"""
inp[105] = f""" R_SOURCE=  {res*2}     ,\n"""
inp[106] = f""" VEL_SOURCE= {float(volume)/(radius**2*np.pi*600)} \n"""
h.seek(0)
h.writelines(inp)
h.truncate()
h.close()

subprocess.run(f"""ln -s ../{dem} topography_dem.asc""",shell=True)
p = subprocess.Popen(['./IMEX_LavaFlow'], stdin=subprocess.PIPE, shell=True)
p.communicate(input=b'\n')

In [ ]:
os.chdir('/home/jovyan/Lava Benchmark/lava2d')

victor.create_geotiff(
    data=stacked,
    output_path='Benchmark_split.tiff',
    x_min=1000,  # easting
    y_min=1000,  # northing
    pixel_width=.03, 
    pixel_height=-.03,  # negative because image coordinates go down
    crs='EPSG:32610',  # UTM Zone 10N example
    nodata_value=-9999
)
r = rxr.open_rasterio("Benchmark_split.tiff",masked=True)
for i in range(14):
    j = int(np.round(np.sin(np.deg2rad(split_angle))))
    r[0,y+j,x+j] = 5000
    r[0,y-j,x+j] = 5000
r.rio.to_raster('Benchmark_split.tiff')
lonlat = r.rio.reproject("EPSG:4326")
xmin,ymin,xmax,ymax = lonlat.rio.bounds()
center = float(lonlat[:,150,115].x), float(lonlat[:,150,115].y)

k=open("./input.py","r+")
inp = k.readlines()
inp[4] = '    path_to_dem_file    = ("./Benchmark_split.tiff"),\n'
inp[5] = f"""    Lon_SRC             = {center[0]}, # source longitude\n"""
inp[6] = f"""    Lat_SRC             = {center[1]},    # source latitude\n"""
inp[7] = f"""    Lon_LowerLeft       = {xmin}, # bounding box: lower-left longitude\n"""
inp[8] = f"""    Lat_LowerLeft       = {ymin}, # bounding box: lower-left latitude\n"""
inp[9] = f"""    Lon_UpperRight      = {xmax}, # bounding box: upper-right longitude\n"""
inp[10] = f"""    Lat_UpperRight      = {ymax},   # bounding box: upper-right latitude\n"""
inp[67] = f"""    max_time_hr = {600/3600},\n"""

k.seek(0)
k.writelines(inp)
k.truncate()
k.close()

k=open("./example_vents/vent_01.txt","r+")
inp = k.readlines()
inp[1] = f'''0	{-.03*scale}	{-.03*scale}	{.03*scale}	{.03*scale}	{.01*scale}	{7.7e-4*scale}\n'''

k.seek(0)
k.writelines(inp)
k.truncate()
k.close()

subprocess.run("python input.py", shell=True)

In [ ]:
data = xr.open_dataset("lava2d/outputs/out.nc",group="DATA/PHYSICS",decode_coords="all")
data = data.where(data!=0, np.nan)

coord = xr.open_dataset("lava2d/outputs/out.nc",group="DATA", decode_coords="all")
lat = coord.lat[:,0]
lon = coord.lon[0,:]
data['rows'] = lat.values
data['cols'] = lon.values
data = data.rename({"rows": "y", "cols": "x"})
repeated_arr_x = np.repeat(data.y.values[0], len(data.x))
repeated_arr_y = np.repeat(data.x.values[0], len(data.y))
data.lava_thickness_total.rio.to_raster("lava2d/split_output.tif")

#### The final two cells concern visualization.

The first provides a reference to the final timestamp of each simulation, though all outputs are avilable in their models' respective folder. The second and last cell plots each model side by side. We encourage users to edit this to their personal preference.

In [ ]:
os.chdir("/home/jovyan/Lava Benchmark/")

flowy_outputs = glob.glob(f'flowy/output/Inclined*thickness_masked*.asc')
flowy_out = flowy_outputs[-1]

r = rxr.open_rasterio("IMEX_LavaFlow/Split_max.asc",masked=True)
r2 = rxr.open_rasterio("Molasses/molasses_split.asc",masked=True)
r3 = rxr.open_rasterio(flowy_out,masked=True)
r4 = rxr.open_rasterio("lava2d/split_output.tif",masked=True)
r_min, r_max = r.min(), r.max()
r2_min, r2_max = r2.min(), r2.max()
r3_min, r3_max = r3.min(), r3.max()
r4_min, r4_max = r4.min()/1000, r4.max()/1000

maximum = np.max((r_max,r2_max,r3_max,r4_max))
idx_max = np.argmax((r_max,r2_max,r3_max,r4_max))

In [ ]:
fig, ((ax0, ax1), (ax2, ax3)) = plt.subplots(ncols=2, nrows=2,subplot_kw=dict(projection=ccrs.epsg(32628)), figsize = (12,9))

xticks = np.linspace(-.2, 4.81, 5)
yticks = np.linspace(-1.5, 1.51, 5)

fig, ((ax0, ax1), (ax2, ax3)) = plt.subplots(ncols=2, nrows=2, figsize = (12,9))

thickness0 = victor.plot_flow(dem, "Molasses/molasses_split.asc", axes=ax0,zoom=False,title='Molasses',minimum=1e-6,scale=maximum)
thickness1 = victor.plot_flow(dem, flowy_out, axes=ax1, zoom=False,title='MrLavaLoba/Flowy',minimum=1e-6, scale=maximum)
thickness2 = victor.plot_flow(dem, "IMEX_LavaFlow/Split_max.asc", axes=ax2, zoom=False,title='IMEX_LavaFlow', minimum=1e-6, scale=maximum)
thickness3 = victor.plot_flow("lava2d/Benchmark_split.tiff", "lava2d/split_output.tif", axes=ax3 ,title='Lava2D', minimum=1e-10, scale=maximum*100)

cbar_ax = fig.add_axes([.25, 0, 0.5, 0.05])
thickness = [thickness0, thickness1, thickness2, thickness3]
thickness = thickness[idx_max]
fig.colorbar(thickness, cax=cbar_ax, label="Flow Thickness (m)", orientation="horizontal")